In [ ]:
!pip install -q transformers==4.45.0 trl==0.11.0 \
             datasets==2.19.0 accelerate==0.34.0 sentencepiece scipy lm-eval

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 68.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 95.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [8]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force single GPU


import torch
import torch.nn as nn
import gc, json, math
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

import lm_eval
from lm_eval.models.huggingface import HFLM

MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
OUTPUT_DIR  = "/kaggle/working/tinyllama-hmatrix-addon"
MAX_SEQ_LEN = 512
BATCH_SIZE  = 2
GRAD_ACCUM  = 16
LR          = 1e-4
EPOCHS      = 1
RANK        = 64
LEVEL   = 2
MIN_DIM     = 64
DROPOUT = 0.05
DEVICE      = "cuda:0"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"rank={RANK}  min_dim={MIN_DIM}  → ~3.39M trainable params")
print(f"LoRA rank=32 reference → 3.60M trainable params")

GPU: Tesla T4
rank=64  min_dim=64  → ~3.39M trainable params
LoRA rank=32 reference → 3.60M trainable params


In [9]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# USV Block (leaf)
# -------------------------
class USVBlock(nn.Module):
    def __init__(self, out_dim, in_dim, rank, dropout=0.0):
        super().__init__()
        r = min(rank, min(out_dim, in_dim))

        # identical to lora_A
        self.V = nn.Parameter(torch.empty(r, in_dim))
        nn.init.kaiming_uniform_(self.V, a=math.sqrt(5))

        # identical to lora_B — zero init guarantees ΔW=0 at start
        self.U = nn.Parameter(torch.zeros(out_dim, r))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x):
        return self.dropout(x) @ self.V.T @ self.U.T
# -------------------------
# Recursive H-Matrix Block
# -------------------------
class HMatrixBlock(nn.Module):
    def __init__(self, out_dim, in_dim, rank, level, min_dim=64, dropout=0.0):
        super().__init__()
        self.is_leaf = (level == 0 or min(out_dim, in_dim) <= min_dim)

        if self.is_leaf:
            self.block = USVBlock(out_dim, in_dim, rank, dropout)
            return

        self.in_mid  = in_dim  // 2
        self.out_mid = out_dim // 2

        self.A11 = HMatrixBlock(self.out_mid, self.in_mid, rank, level-1, min_dim, dropout)
        self.A22 = HMatrixBlock(self.out_mid, self.in_mid, rank, level-1, min_dim, dropout)
        self.A12 = USVBlock(self.out_mid, self.in_mid, rank, dropout)
        self.A21 = USVBlock(self.out_mid, self.in_mid, rank, dropout)

    def forward(self, x):
        if self.is_leaf:
            return self.block(x)

        x1 = x[..., :self.in_mid]
        x2 = x[..., self.in_mid:]

        y1 = self.A11(x1) + self.A12(x2)
        y2 = self.A21(x1) + self.A22(x2)

        return torch.cat([y1, y2], dim=-1)


# -------------------------
# H-Matrix Addon
# -------------------------
class HMatrixAddon(nn.Module):
    def __init__(self, out_dim, in_dim, rank, level=1, min_dim=64, dropout=0.0):
        super().__init__()
        self.hblock = HMatrixBlock(out_dim, in_dim, rank, level, min_dim, dropout)

    def forward(self, x):
        shape = x.shape
        return self.hblock(x.reshape(-1, shape[-1])).reshape(*shape[:-1], -1)


# -------------------------
# H-Matrix Linear (FINAL)
# -------------------------
class HMatrixLinear(nn.Module):
    def __init__(self, original_linear, rank, level=1, min_dim=64, dropout=0.0):
        super().__init__()

        out_dim, in_dim = original_linear.weight.shape
        dtype = original_linear.weight.dtype

        self.weight = nn.Parameter(
            original_linear.weight.data.clone(),
            requires_grad=False
        )
        self.bias = (
            nn.Parameter(original_linear.bias.data.clone(), requires_grad=False)
            if original_linear.bias is not None else None
        )

        self.h_addon = HMatrixAddon(out_dim, in_dim, rank, level, min_dim, dropout).to(dtype)

        self.rank  = rank
        self.alpha = 2 * rank
        self.scale = self.alpha / self.rank  # = 2

    def forward(self, x):
        base = F.linear(x, self.weight, self.bias)
        return base + self.scale * self.h_addon(x)


print("✅ H-matrix updated:")
print(" - kaiming_uniform V, zero-init S and U (matches LoRA init exactly)")
print(" - dropout threaded through all levels as param")
print(" - dropout=0.0 default for fair no-dropout comparison with LoRA")

✅ H-matrix updated:
 - kaiming_uniform V, zero-init S and U (matches LoRA init exactly)
 - dropout threaded through all levels as param
 - dropout=0.0 default for fair no-dropout comparison with LoRA


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)

# Freeze all original params
for p in model.parameters():
    p.requires_grad = False

# Replace Q, K, V with HMatrixLinear (frozen original + trainable addon)
for i, layer in enumerate(model.model.layers):
    attn = layer.self_attn
    attn.q_proj = HMatrixLinear(attn.q_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)
    attn.k_proj = HMatrixLinear(attn.k_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)
    attn.v_proj = HMatrixLinear(attn.v_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)


trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({trainable/1e6:.2f}M)")
print(f"Total    : {total_p:,} ({total_p/1e6:.2f}M)")
print(f"\nLoRA reference: 3.60M trainable")
print(f"H-matrix addon: {trainable/1e6:.2f}M trainable")

Trainable: 36,765,696 (36.77M)
Total    : 1,136,814,080 (1136.81M)

LoRA reference: 3.60M trainable
H-matrix addon: 36.77M trainable


In [11]:
def fmt(text):
    return str(text).strip()

def format_medqa(ex):
    opts = ex['options']
    choices = "\n".join([f"({k}) {v}" for k, v in opts.items()])
    
    prompt = (
        f"Question: {fmt(ex['question'])}\n\n"
        f"{choices}\n\n"
        f"Answer: ("
    )
    answer = f"{fmt(ex['answer_idx'])})"
    
    return {
        "prompt": prompt,
        "answer": answer,
        "text": prompt + answer   # full text for tokenization
    }

medqa_train = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
medqa_val   = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
medqa_train = medqa_train.map(format_medqa, remove_columns=medqa_train.column_names)
medqa_val   = medqa_val.map(format_medqa,   remove_columns=medqa_val.column_names)

print(f"Train: {len(medqa_train):,}")
print(f"Val  : {len(medqa_val):,}")
print(f"\nSample:\n{medqa_train[0]['text']}")

Train: 10,178
Val  : 1,273

Sample:
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

(A) Ampicillin
(B) Ceftriaxone
(C) Doxycycline
(D) Nitrofurantoin

Answer: (D)


In [12]:
def make_collator(tokenizer, max_len):
    def collate(batch):
        full_texts = [b["text"] for b in batch]
        
        enc = tokenizer(full_texts, truncation=True, max_length=max_len,
                        padding="max_length", return_tensors="pt")
        
        labels = enc["input_ids"].clone()

        for i, text in enumerate(full_texts):
            split_marker = "Answer: ("
            prompt_part  = text[:text.rfind(split_marker) + len(split_marker)]
            prompt_ids   = tokenizer(prompt_part, add_special_tokens=False)["input_ids"]
            prompt_len   = len(prompt_ids)
            labels[i, :prompt_len] = -100

        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels
        return enc
    return collate

In [13]:
from transformers import Trainer

model.enable_input_require_grads()

training_args = TrainingArguments(
    save_total_limit=1,
    output_dir="./output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    group_by_length=False,
    optim="adamw_torch",
    weight_decay=0.01,
    remove_unused_columns=False,  # keep prompt/answer/text columns for collator
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=make_collator(tokenizer, MAX_SEQ_LEN),
    train_dataset=medqa_train,
    eval_dataset=medqa_val,
    args=training_args,
)

trainer.train()
print("Done!")

Step,Training Loss,Validation Loss
100,0.467200,0.463170
200,0.462500,0.465684
300,0.462700,0.463477


Done!


In [14]:
from lm_eval.models.huggingface import HFLM
import lm_eval
lm_model_ft = HFLM(
    pretrained=trainer.model,
    tokenizer=tokenizer,
    batch_size=8,
)

results_after = lm_eval.simple_evaluate(
    model=lm_model_ft,
    tasks=["medqa_4options", "pubmedqa"],
)

medqa_after  = results_after["results"]["medqa_4options"]["acc,none"] * 100
pubmed_after = results_after["results"]["pubmedqa"]["acc,none"] * 100

print(f"  medqa_4options : {medqa_after:.2f}  (baseline: 24.74)")
print(f"  pubmedqa       : {pubmed_after:.2f}  (baseline: 60.80)")

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1272 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for bigbio/pubmed_qa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/pubmed_qa
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Running loglikelihood requests: 100%|██████████| 6592/6592 [08:57<00:00, 12.26it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


  medqa_4options : 28.44  (baseline: 24.74)
  pubmedqa       : 62.80  (baseline: 60.80)


In [ ]:
print("test")